# Coupled Pendulums Without Damping: Differential Equations and Physics-Informed Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nhungnh-dot/pendulumDE/blob/main/coupled_pendulum_no_damping_lesson.ipynb)

> by Andrew Bennett and Nhung Nguyen

Study undamped coupled ODEs, compare predictions with experimental data, and estimate an unknown spring coupling using a PINN.

## 1. Model and coupling

Let

- $\theta_1(t)$ = angle of the **left** pendulum,
- $\theta_2(t)$ = angle of the **right** pendulum.

The two pendulums have lengths $l_1$ and $l_2$. They are connected by a spring, so the motion of one pendulum affects the other.

For small angles, we use the approximation

$$
\sin(\theta)\approx \theta.
$$

In this version of the lesson, we ignore damping. The equations of motion are

$$
\theta_1''
+\frac{g}{l_1}\theta_1
+\frac{K}{l_1}(l_1\theta_1-l_2\theta_2)
=0,
$$

$$
\theta_2''
+\frac{g}{l_2}\theta_2
-\frac{K}{l_2}(l_1\theta_1-l_2\theta_2)
=0.
$$

Here a prime means differentiation with respect to time. For example,

$$
\theta_1'=\frac{d\theta_1}{dt},
\qquad
\theta_1''=\frac{d^2\theta_1}{dt^2}.
$$

**What do the terms mean?**

Each equation contains familiar pieces:

- $\theta_i''$: **angular acceleration**;
- $\dfrac{g}{l_i}\theta_i$: the restoring effect of **gravity**;
- the term containing $K$: the effect of the **spring connecting the two pendulums**.

The constant

$$
K=\frac{k}{m}
$$

combines the spring constant $k$ and the pendulum-bob mass $m$. We assume the two bobs have the same mass.

Because the equations contain both $\theta_1$ and $\theta_2$, they are **coupled**. We cannot solve for one pendulum without also considering the other.

We will estimate both angles and $K$ by matching experimental data and approximately satisfying the ODEs.

**Record your work**

Use the questions, notes, and tables in this notebook to guide your investigation. You may write your responses and numerical results on your own paper for submission. Follow the **Canvas assignment page** for exactly what to submit and how to submit it.

> **❓ Question 1: What Does Coupling Do?**
>
> *Set $K=0$ and write the simplified equations. What disappears, and why does this mean the pendulums move independently?*

## 2. Experimental data and setup

Initialize PyTorch with **seed 7**, then load the CSV: time and coordinates for both pendulums. Use horizontal coordinates `a_left` and `a_right`; remove header/invalid rows, convert to numbers, and sort by time.

Use $l_1=1.00$ m, $l_2=1.05$ m, and $g=9.8$ m/s$^2$, with angle 1 for the left pendulum and angle 2 for the right. Shift time by its first value so the clock starts at zero without changing intervals.

For small angles, $x=L\sin\theta\approx L\theta$, so convert positions using $\theta_1\approx a_{\mathrm{left}}/l_1$ and $\theta_2\approx a_{\mathrm{right}}/l_2$. Inspect the measured oscillations before fitting.


In [ ]:
# ============================================================
# COUPLED PENDULUM PINN
# Experimental data + Fourier features + gradual physics
#
# Google Colab / PyTorch
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
# Resetting the seed before each new model is constructed makes repeated
# experiments start from the same neural-network initialization.
import random

SEED = 7

def reset_random_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

reset_random_seed()

# Reduce run-to-run variation on CUDA when possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
# ============================================================
# LOAD EXPERIMENTAL DATA
# ============================================================

csv_url = (
    "https://raw.githubusercontent.com/"
    "ngochuytruong/data/main/data-diff-length.csv"
)
df = pd.read_csv(csv_url, header=None)

df.columns = [
    't',
    'a_right',
    'b_right',
    'a_left',
    'b_left'
]

# Remove text/header rows in original CSV
df = df.iloc[2:].reset_index(drop=True)

# Convert everything to numeric
df = df.apply(
    pd.to_numeric,
    errors='coerce'
)

# Remove invalid rows
df = df.dropna().reset_index(drop=True)

# Sort by time just in case
df = df.sort_values('t').reset_index(drop=True)

print(df.head())
print()
print("Number of experimental points:", len(df))

# ============================================================
# EXTRACT POSITION DATA
# ============================================================

t = df['t'].to_numpy(dtype=np.float32)

a_right = df['a_right'].to_numpy(dtype=np.float32)
a_left  = df['a_left'].to_numpy(dtype=np.float32)

# Pendulum lengths [m]
l_left  = 1.00
l_right = 1.05

# Equation notation:
#
# theta1 = left
# theta2 = right
#
l1 = l_left
l2 = l_right

g = 9.8

# ============================================================
# SHIFT TIME TO START AT ZERO
# ============================================================

t = t - t[0]

# ============================================================
# CONVERT POSITION -> ANGLE
# ============================================================
#
# We assume a_left and a_right are horizontal x positions.
#
# First remove the equilibrium offset.
#
# For horizontal displacement:
#
#       x = L sin(theta)
#
# therefore
#
#       theta = arcsin(x/L)
#
# If your columns already represent displacement from x=0,
# subtracting the median will have very little effect.
#
# ============================================================

x_left = a_left
x_right = a_right

theta_left = x_left / l_left
theta_right = x_right / l_right

print("Theta left range:",
      theta_left.min(),
      theta_left.max())

print("Theta right range:",
      theta_right.min(),
      theta_right.max())

# Plot
plt.figure(figsize=(11, 5))

plt.plot(
    t,
    theta_left,
    label='Left experimental'
)

plt.plot(
    t,
    theta_right,
    '--',
    label='Right experimental'
)

plt.xlabel('Time [s]')
plt.ylabel(r'$\theta$ [rad]')
plt.title('Experimental Coupled Pendulum')
plt.legend()
plt.grid(True)

plt.show()

**Chronological training/validation split**

We will not use every measured point to fit the model.

Instead, we divide the measurements by time:

- the **first 80%** are training data;
- the **last 20%** are validation data.

**Why do this?**

The training data are the observations that the model is allowed to learn from.

The validation data are kept aside. After training, we ask:

> Can the model predict the later part of the experiment even though those measured angles were not used to fit it?

Because the last 20% occur later in time, this is a test of **prediction into the future**, or extrapolation.

The split is chronological rather than random:

$$
\boxed{\text{earlier times: training}}
\qquad
\boxed{\text{later times: validation}}.
$$

The known differential equations will still be enforced over the full time interval. However, the measured validation angles themselves are not used in the fitting loss.

The code also defines a typical angle size, called `theta_scale`, using only the training data. This scale helps keep numerical values at a reasonable size during training.

The split plots mark the validation boundary and shade the final 20% of the experiment.


In [ ]:
# ============================================================
# PREPARE PYTORCH DATA + CHRONOLOGICAL 80/20 SPLIT
# ============================================================

t_data = t.reshape(-1, 1)
theta1_data = theta_left.reshape(-1, 1)
theta2_data = theta_right.reshape(-1, 1)

t_min = float(t_data.min())
t_max = float(t_data.max())

# ------------------------------------------------------------
# First 80% of observations for training; last 20% for validation.
# df was sorted by t during preprocessing. NO random shuffling.
# ------------------------------------------------------------
train_fraction = 0.80
n_data = len(t_data)
assert n_data >= 2, "At least two observations are required to split the data."
assert np.all(np.diff(t_data[:, 0]) >= 0), "Time must be sorted."

n_train = int(np.floor(train_fraction * n_data))
train_indices = np.arange(n_train)
val_indices = np.arange(n_train, n_data)

# NumPy arrays used for plots and preprocessing.
t_train = t_data[train_indices]
t_val = t_data[val_indices]
theta1_train = theta1_data[train_indices]
theta2_train = theta2_data[train_indices]
theta1_val = theta1_data[val_indices]
theta2_val = theta2_data[val_indices]

# First held-out time marks the beginning of the future validation window.
train_time_end = float(t_train[-1, 0])
split_time = float(t_val[0, 0])

assert len(train_indices) + len(val_indices) == n_data
assert len(np.intersect1d(train_indices, val_indices)) == 0
assert train_time_end <= split_time

# ------------------------------------------------------------
# Characteristic angle scale: TRAINING ANGLES ONLY.
# ------------------------------------------------------------
theta_scale = max(
    np.max(np.abs(theta1_train)),
    np.max(np.abs(theta2_train)),
)
theta_scale = max(theta_scale, 1e-3)

# PyTorch tensors used by the network.
t_train_torch = torch.tensor(t_train, dtype=torch.float32, device=device)
t_val_torch = torch.tensor(t_val, dtype=torch.float32, device=device)
theta1_train_torch = torch.tensor(theta1_train, dtype=torch.float32, device=device)
theta2_train_torch = torch.tensor(theta2_train, dtype=torch.float32, device=device)
theta1_val_torch = torch.tensor(theta1_val, dtype=torch.float32, device=device)
theta2_val_torch = torch.tensor(theta2_val, dtype=torch.float32, device=device)

# The full time tensor is for evaluation and plotting ONLY.
t_data_torch = torch.tensor(t_data, dtype=torch.float32, device=device)

print("Angle scale (training only):", theta_scale)
print(f"Total observations:      {n_data}")
print(f"Training observations:   {len(train_indices)} ({len(train_indices)/n_data:.1%})")
print(f"Validation observations: {len(val_indices)} ({len(val_indices)/n_data:.1%})")
print(f"Training time window:    [{t_min:.3f}, {train_time_end:.3f}] s")
print(f"Validation time window:  [{split_time:.3f}, {t_max:.3f}] s")


In [ ]:
# ============================================================
# 7B. VISUALIZE THE TRAINING / VALIDATION SPLIT
# ============================================================

# Left pendulum
plt.figure(figsize=(12, 4.5))

plt.plot(
    t_data[:, 0],
    theta1_data[:, 0],
    linewidth=1,
    alpha=0.25,
    label="All measured data"
)

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=12,
    alpha=0.50,
    label=f"Training ({len(train_indices)} points)"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label=f"Validation ({len(val_indices)} points)"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Left Pendulum: 80/20 Training–Validation Split")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Right pendulum
plt.figure(figsize=(12, 4.5))

plt.plot(
    t_data[:, 0],
    theta2_data[:, 0],
    linewidth=1,
    alpha=0.25,
    label="All measured data"
)

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=12,
    alpha=0.50,
    label=f"Training ({len(train_indices)} points)"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label=f"Validation ({len(val_indices)} points)"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Right Pendulum: 80/20 Training–Validation Split")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Network and losses

Up to this point, everything has been standard data preparation and differential-equation modeling.

We now introduce the main computational tool: a **Physics-Informed Neural Network**, or PINN.

You do not need a machine-learning course to follow the main idea.

A neural network is simply being used here as a flexible function. Instead of guessing a formula such as

$$
\theta(t)=A\cos(\omega t)+B\sin(\omega t),
$$

we let the computer build flexible approximations

$$
\widehat{\theta}_1(t)
\quad\text{and}\quad
\widehat{\theta}_2(t).
$$

The hat means "predicted."

**Rescaling time**

The physical time values are first converted to a new variable $\tau$ between $-1$ and $1$:

$$
\tau
=
2\frac{t-t_{\min}}{t_{\max}-t_{\min}}-1.
$$

This does not change the physical problem. It is only a numerical rescaling.

**Why add sine and cosine terms?**

Pendulums oscillate. Because sine and cosine functions naturally describe oscillations, we give the network extra inputs such as

$$
\sin(\pi\tau),\quad
\sin(2\pi\tau),\quad
\cos(\pi\tau),\quad
\cos(2\pi\tau),\ldots
$$

These are called **Fourier features**.

The number of Fourier frequencies is controlled by the single student setting `N_FREQ`. With `N_FREQ = 30`, the network receives

$$
\tau,
\sin(n\pi\tau),
\cos(n\pi\tau),
\qquad n=1,\ldots,30.
$$

When you experiment with the number of Fourier features later, change only `N_FREQ` in the marked code cell. The purpose is not to change the differential equation. The Fourier features simply make it easier for the neural network to represent oscillatory functions.

The network uses **64 neurons in each hidden layer**. Rebuilding it resets **seed 7**, making repeated runs of a given architecture reproducible. Changing `N_FREQ` changes the input dimensions.

Finally, the network returns two numbers for each time $t$:

$$
\begin{bmatrix}
\widehat{\theta}_1(t)\\
\widehat{\theta}_2(t)
\end{bmatrix}.
$$

So you can think of the neural network as our computer-generated approximation to the two unknown solution functions.


In [ ]:
# ============================================================
# FOURIER FEATURE PINN
# ============================================================


# ============================================================
# STUDENT SETTING: NUMBER OF FOURIER FREQUENCIES
# Change ONLY this value for the Fourier-feature experiment.
# ============================================================
N_FREQ = 30


class FourierPINN(nn.Module):

    def __init__(
        self,
        t_min,
        t_max,
        theta_scale,
        n_freq,
        hidden=64
    ):

        super().__init__()

        self.register_buffer(
            "t_min",
            torch.tensor(
                float(t_min),
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "t_max",
            torch.tensor(
                float(t_max),
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "theta_scale",
            torch.tensor(
                float(theta_scale),
                dtype=torch.float32
            )
        )

        frequencies = torch.arange(
            1,
            n_freq + 1,
            dtype=torch.float32
        )

        self.register_buffer(
            "frequencies",
            frequencies
        )

        # tau + sin features + cos features
        input_dim = 1 + 2*n_freq

        self.net = nn.Sequential(

            nn.Linear(input_dim, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, 2)
        )


    def forward(self, t):

        # -----------------------------------
        # Normalize physical time to [-1, 1]
        # -----------------------------------

        tau = (
            2.0
            * (t - self.t_min)
            / (self.t_max - self.t_min)
            - 1.0
        )


        # -----------------------------------
        # Fourier features
        # -----------------------------------

        phase = (
            np.pi
            * tau
            * self.frequencies.reshape(1, -1)
        )


        features = torch.cat(
            (
                tau,
                torch.sin(phase),
                torch.cos(phase)
            ),
            dim=1
        )


        # Network predicts dimensionless angle
        raw_output = self.net(features)

        # Convert back to physical radians
        output = self.theta_scale * raw_output

        return output


# Reset the seed so re-running this cell gives the same initialization.
reset_random_seed()

model = FourierPINN(
    t_min=t_min,
    t_max=t_max,
    theta_scale=theta_scale,
    n_freq=N_FREQ,
    hidden=64
).to(device)


print()
print(model)

**Unknown physical parameter**

The differential equations contain one quantity that we want to estimate:

$$
K=\frac{k}{m}.
$$

The parameter $K$ measures the strength of the spring coupling per unit mass.

The computer will adjust this value while it adjusts the predicted functions $\widehat{\theta}_1$ and $\widehat{\theta}_2$.

**Why force the parameter to be positive?**

For this model, a negative spring strength would not have the physical meaning we want. Therefore, the code represents $K$ using a positive transformation called the **softplus** function:

$$
\operatorname{softplus}(x)=\log(1+e^x).
$$

Since

$$
\operatorname{softplus}(x)>0,
$$

the estimate of $K$ remains positive during training.

The starting guess is

$$
K=1.0.
$$

This is only a starting value. The computer will update it later.

For this lesson, the key point is:

> The physical constant is treated as an **unknown parameter in the differential equations**, and we estimate it from the observed motion.


> **❓ Question 2: What Does the Coupling Constant Tell Us?**
>
> Our model can predict future motion and recover the unknown coupling constant, $K=k/m$.
>
> *How would a stiffer spring or heavier bobs change $K$ and the coupling’s effect on the motion?*



In [ ]:
# ============================================================
# TRAINABLE PHYSICAL PARAMETER
# ============================================================
#
# Instead of assuming k = 1, estimate k/m.
#
# Let
#
#     K = k/m
#
# softplus guarantees:
#
#     K > 0
#
# ============================================================

def inverse_softplus(x):

    x = torch.tensor(
        x,
        dtype=torch.float32,
        device=device
    )

    return torch.log(
        torch.exp(x) - 1.0
    )


# Reasonable starting guess

raw_K = nn.Parameter(
    inverse_softplus(1.0)
)


def physical_parameters():

    K = F.softplus(raw_K)

    return K


**ODE residuals**

This is where the differential equations become part of the learning process.

The neural network gives predicted functions

$$
\widehat{\theta}_1(t),
\qquad
\widehat{\theta}_2(t).
$$

PyTorch can differentiate these functions automatically, so it can compute

$$
\widehat{\theta}_1',
\quad
\widehat{\theta}_1'',
\quad
\widehat{\theta}_2',
\quad
\widehat{\theta}_2''.
$$

Now substitute the predictions into the two differential equations.

For the left pendulum, define

$$
R_1(t)
=
\widehat{\theta}_1''
+\frac{g}{l_1}\widehat{\theta}_1
+\frac{K}{l_1}
(l_1\widehat{\theta}_1-l_2\widehat{\theta}_2).
$$

For the right pendulum,

$$
R_2(t)
=
\widehat{\theta}_2''
+\frac{g}{l_2}\widehat{\theta}_2
-\frac{K}{l_2}
(l_1\widehat{\theta}_1-l_2\widehat{\theta}_2).
$$

The quantities $R_1$ and $R_2$ are called **residuals**.

If the predicted functions satisfied the ODEs exactly, then

$$
R_1(t)=0,
\qquad
R_2(t)=0
$$

for every time $t$.

So a small residual means that the predicted motion is close to obeying the differential-equation model.


In [ ]:
# ============================================================
# AUTOMATIC DIFFERENTIATION + PHYSICS
# ============================================================

def physics_residual(model, t):

    # Need derivatives with respect to physical t
    t = t.clone().detach().requires_grad_(True)

    output = model(t)

    theta1 = output[:, 0:1]
    theta2 = output[:, 1:2]


    # ========================================================
    # First derivatives
    # ========================================================

    theta1_t = torch.autograd.grad(
        theta1,
        t,
        grad_outputs=torch.ones_like(theta1),
        create_graph=True
    )[0]

    theta2_t = torch.autograd.grad(
        theta2,
        t,
        grad_outputs=torch.ones_like(theta2),
        create_graph=True
    )[0]


    # ========================================================
    # Second derivatives
    # ========================================================

    theta1_tt = torch.autograd.grad(
        theta1_t,
        t,
        grad_outputs=torch.ones_like(theta1_t),
        create_graph=True
    )[0]

    theta2_tt = torch.autograd.grad(
        theta2_t,
        t,
        grad_outputs=torch.ones_like(theta2_t),
        create_graph=True
    )[0]


    # ========================================================
    # Physical parameter
    # ========================================================

    K = physical_parameters()


    # ========================================================
    # Spring extension
    #
    # l1 theta1 - l2 theta2
    # ========================================================

    coupling = (
        l1 * theta1
        -
        l2 * theta2
    )


    # ========================================================
    # ODE #1
    #
    # theta1''
    # + g/l1 theta1
    # + K/l1 (l1 theta1-l2 theta2)
    # = 0
    # ========================================================

    residual1 = (
        theta1_tt
        +
        (g/l1) * theta1
        +
        (K/l1) * coupling
    )


    # ========================================================
    # ODE #2
    # ========================================================

    residual2 = (
        theta2_tt
        +
        (g/l2) * theta2
        -
        (K/l2) * coupling
    )


    return (
        residual1,
        residual2,
        theta1,
        theta2,
        theta1_t,
        theta2_t
    )


**Data and validation loss**

We need a numerical way to measure the difference between the model prediction and the experimental measurements. This is the **data loss**.

At each training time $t_j$, the neural network predicts

$$
\widehat{\theta}_1(t_j),
\qquad
\widehat{\theta}_2(t_j),
$$

and we compare those predictions with the measured angles

$$
\theta_{1,j},
\qquad
\theta_{2,j}.
$$

For each pendulum, we first compute the prediction error. For example, for the left pendulum,

$$
\widehat{\theta}_1(t_j)-\theta_{1,j}.
$$

If this difference is small, the prediction is close to the measurement. We square the differences so that positive and negative errors do not cancel each other, and then average them over all training points.

In simplified form,

$$
L_{\text{data}}
=
\frac{1}{N}\sum_j
\left(
\frac{\widehat{\theta}_1(t_j)-\theta_{1,j}}
{\theta_{\text{scale}}}
\right)^2
+
\frac{1}{N}\sum_j
\left(
\frac{\widehat{\theta}_2(t_j)-\theta_{2,j}}
{\theta_{\text{scale}}}
\right)^2.
$$

So the data loss asks one main question:

> **How closely does the neural network match the measured motion?**

A small data loss means the predicted curves are close to the experimental points. A large data loss means the predictions are still far from the measurements.

The division by $\theta_{\text{scale}}$ simply rescales the error so that the loss is dimensionless and easier to work with numerically.

Only the **first 80% of the measurements** are used in this training loss. The final 20% are not used to update the network. Instead, we compute the same type of error on those held-out points and call it the **validation loss**.

The validation loss combines the errors from **both the left and right pendulums** into one number. It is used only to evaluate how well the model predicts data it did not use directly for training.


In [ ]:
# ============================================================
# TRAINING + VALIDATION DATA LOSS
# ============================================================

def normalized_data_mse(
    t_tensor,
    theta1_tensor,
    theta2_tensor
):
    """Dimensionless MSE for any supplied data subset."""

    prediction = model(t_tensor)

    pred1 = prediction[:, 0:1]
    pred2 = prediction[:, 1:2]

    error1 = (
        pred1 - theta1_tensor
    ) / theta_scale

    error2 = (
        pred2 - theta2_tensor
    ) / theta_scale

    return (
        torch.mean(error1**2)
        +
        torch.mean(error2**2)
    )


def data_loss():
    """Training loss: used for backpropagation."""

    return normalized_data_mse(
        t_train_torch,
        theta1_train_torch,
        theta2_train_torch
    )


def validation_loss():
    """Held-out loss: evaluation only, never backpropagated."""

    with torch.no_grad():
        return normalized_data_mse(
            t_val_torch,
            theta1_val_torch,
            theta2_val_torch
        )


**Physics loss**

The residuals above tell us how far the predicted motion is from satisfying the two ODEs. We square and average them at **collocation points**: times chosen to check the differential equations, which need not coincide with measurements. This lets physics guide the curves between measured points and over the later validation interval without using the held-out angles.

For each call, the code samples 512 times over the full interval. To make the loss dimensionless, divide each residual by a characteristic angular-acceleration scale before squaring:
$$
L_{\mathrm{physics}}=
\operatorname{mean}\left[\left(\frac{R_1}{(g/l_1)\theta_{\mathrm{scale}}}\right)^2\right]
+\operatorname{mean}\left[\left(\frac{R_2}{(g/l_2)\theta_{\mathrm{scale}}}\right)^2\right].
$$

A small physics loss means that the predicted curves approximately obey the undamped model. This answers a different question from data loss: **does the motion satisfy the equations, rather than merely pass close to measured points?** A flexible network can fit angles well while having derivatives that violate the ODEs. Stage 2 therefore uses both losses.


In [ ]:
# ============================================================
# PHYSICS LOSS
# ============================================================

def physics_loss(
    n_physics=512
):

    # Generate new random collocation points every iteration.
    #
    # This is usually better and cheaper than using thousands
    # of permanently fixed points.

    # Known ODEs can be enforced at future times without using
    # the held-out future MEASUREMENTS. This is physics-assisted
    # forecasting, not training on validation labels.
    # For the stricter training-time-only-physics experiment,
    # replace t_max below with train_time_end.
    physics_time_end = t_max
    t_phys = (
        t_min
        +
        (physics_time_end - t_min)
        * torch.rand(
            n_physics,
            1,
            device=device
        )
    )


    r1, r2, _, _, _, _ = physics_residual(
        model,
        t_phys
    )


    # Characteristic acceleration:
    #
    #       (g/L) theta_scale
    #
    # Divide residual by this so the physics loss is
    # dimensionless.

    acceleration_scale_1 = (
        (g/l1) * theta_scale
    )

    acceleration_scale_2 = (
        (g/l2) * theta_scale
    )


    r1_normalized = (
        r1 / acceleration_scale_1
    )

    r2_normalized = (
        r2 / acceleration_scale_2
    )


    loss = (
        torch.mean(r1_normalized**2)
        +
        torch.mean(r2_normalized**2)
    )

    return loss

## 4. Stage 1: Data-only training

Training is done in two stages.

In **Stage 1**, we temporarily ignore the differential equations and ask the neural network only to fit the training measurements.

Symbolically,

$$
\text{minimize }L_{\text{data}}.
$$

During this stage:

- the neural-network approximation changes;
- only the first 80% of the measured angles are used for fitting;
- $K$ is not changed yet.

**Why start this way?**

It gives the network a reasonable first approximation to the observed oscillations before we ask it to satisfy the more restrictive physical model.

Think of this as making an initial guess for the solution before adding the ODE information.

The main run uses 10,000 epochs.


In [ ]:
# ============================================================
# STAGE 1:
#     DATA-ONLY PRETRAINING ON TRAINING SET
# ============================================================

warmup_epochs = 10000

optimizer_data = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

warmup_history = []

print()
print("========================================")
print("STAGE 1: DATA-ONLY TRAINING")
print("========================================")

for epoch in range(warmup_epochs):

    optimizer_data.zero_grad()

    # Uses only the training observations.
    loss_d = data_loss()

    loss_d.backward()
    optimizer_data.step()

    warmup_history.append(loss_d.item())

    if (epoch + 1) % 1000 == 0:
        print(
            f"Epoch {(epoch+1):5d} | "
            f"Data loss = {loss_d.item():.4e}"
        )

**Evaluate Stage 1 on unseen measurements**

Stage 1 has finished training. **Run the next cell** to compare its final training data loss with one held-out validation loss (combining both pendulums).

The validation loss is evaluated **once after training**, not at every epoch. These held-out measurements do not affect the model's weights. Then inspect the two Stage 1 trajectory plots.

**Check the Stage 1 fit**

After the validation check, the two plots compare the measurements with the neural-network prediction after data-only training.

Look separately at the **left-pendulum plot** and the **right-pendulum plot**. In each plot, distinguish among:

- the training points, which the network used;
- the validation points, which the network did not use;
- the continuous predicted curve.

Questions to consider:

1. **Left-pendulum plot:** How closely does the predicted $\theta_1(t)$ curve follow the training measurements? How well does it match the later validation measurements?
2. **Right-pendulum plot:** How closely does the predicted $\theta_2(t)$ curve follow the training measurements? How well does it match the later validation measurements?
3. Do you notice a difference between the quality of the left- and right-pendulum fits?
4. Do the predicted curves continue to oscillate in a physically reasonable way in the validation interval?

At this stage, the network has learned from data, but we have not yet required it to satisfy the coupled-pendulum ODEs.


In [ ]:
# ============================================================
# STAGE 1: HELD-OUT VALIDATION (AFTER TRAINING ONLY)
# ============================================================
# One evaluation after all training epochs; no validation in optimizer.
model.eval()
with torch.no_grad():
    stage1_train_loss = data_loss().item()
    stage1_val_loss = validation_loss().item()
model.train()
print("========================================")
print("STAGE 1: FINAL TRAINING AND VALIDATION LOSS")
print("========================================")
print(f"Data loss   = {stage1_train_loss:.6e}")
print(f"Validation data loss = {stage1_val_loss:.6e}")


In [ ]:

# ============================================================
# CHECK DATA-ONLY FIT
#     PLOT LEFT AND RIGHT PENDULUMS SEPARATELY
# ============================================================

model.eval()

with torch.no_grad():
    prediction_warmup = (
        model(t_data_torch)
        .cpu()
        .numpy()
    )

model.train()

# ------------------------------------------------------------
# LEFT PENDULUM
# ------------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=10,
    alpha=0.35,
    label="Left training data"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=10,
    label="Left validation data"
)

plt.plot(
    t,
    prediction_warmup[:, 0],
    color="red",
    linewidth=2,
    label="Left NN after warm-up"
)

plt.axvspan(
    split_time, t_max,
    color="gold",
    alpha=0.12,
    label="Future validation window"
)

plt.axvline(
    split_time,
    color="black",
    linestyle=":",
    linewidth=1.4
)

plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Stage 1: Left Pendulum — Training vs Validation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# RIGHT PENDULUM
# ------------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=10,
    alpha=0.35,
    label="Right training data"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=10,
    label="Right validation data"
)

plt.plot(
    t,
    prediction_warmup[:, 1],
    color="red",
    linewidth=2,
    label="Right NN after warm-up"
)

plt.axvspan(
    split_time, t_max,
    color="gold",
    alpha=0.12,
    label="Future validation window"
)

plt.axvline(
    split_time,
    color="black",
    linestyle=":",
    linewidth=1.4
)

plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Stage 1: Right Pendulum — Training vs Validation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

> **❓ Question 3: How Many Fourier Features Do We Need?**
>
> Compare `N_FREQ` values **0, 5, and 30**, changing only the marked setting. For each value, rebuild the network, run Stage 1, then run its validation check and trajectory plots. The seed resets when the network is rebuilt.
>
> | `N_FREQ` | Final Stage 1 training loss | Validation loss after Stage 1 |
> |---:|---:|---:|
> | 0 |  |  |
> | 5 |  |  |
> | 30 |  |  |
>
> **Discuss your observations**
>
> 1. *Which setting gives the lowest training loss and which gives the lowest validation loss? Why needn’t low training loss imply low validation loss on later measurements?*
> 2. *Do more Fourier features improve the later predictions? Choose a setting using your losses and plots.*
>
**Before continuing:** Set your chosen `N_FREQ`, rebuild the network, and re-run Stage 1 and its validation check.

## 5. Stage 2: Physics-informed training

Now ask the predicted motion to meet both goals: match the measurements and approximately satisfy the ODEs. Combine the losses:
$
L_{\mathrm{total}}=L_{\mathrm{data}}+\lambda_pL_{\mathrm{physics}}.
$

The weight $\lambda_p$ controls how strongly the equations influence training. The default is **0.5**; Question 4 compares other choices. Increasing it gives ODE agreement more influence relative to fitting the data, but does not guarantee better predictions of later measurements.

During these 10,000 epochs, the optimizer updates **both the network and the coupling parameter $K$**. Stage 1 supplies an initial approximation to the motion; Stage 2 adjusts it using the undamped physical model. The measured validation angles remain held out.


In [ ]:
# ============================================================
# STAGE 2:
#     JOINT TRAINING DATA + PHYSICS
# ============================================================

physics_epochs = 10000


optimizer_pinn = torch.optim.Adam(
    list(model.parameters())
    +
    [raw_K],
    lr=5e-4
)

total_history = []
data_history = []
physics_history = []
K_history = []

# ============================================================
# STUDENT SETTING: PHYSICS-LOSS WEIGHT
# Default value for the main run. Change this only when Question 4 asks you to.
# ============================================================
lambda_p = 0.5

print()
print("========================================")
print("STAGE 2: PINN TRAINING")
print("========================================")

for epoch in range(physics_epochs):

    optimizer_pinn.zero_grad()

    # ----------------------------------------
    # Training observations only
    # ----------------------------------------
    loss_d = data_loss()

    # ----------------------------------------
    # Physics over the full horizon (NO future measured angles)
    # ----------------------------------------
    loss_p = physics_loss(
        n_physics=512
    )

    total_loss = (
        loss_d
        +
        lambda_p * loss_p
    )

    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(model.parameters())
        +
        [raw_K],
        max_norm=10.0
    )

    optimizer_pinn.step()

    # ----------------------------------------
    # Save training histories
    # ----------------------------------------
    K_now = physical_parameters()

    total_history.append(total_loss.item())
    data_history.append(loss_d.item())
    physics_history.append(loss_p.item())

    K_history.append(K_now.item())


    # ----------------------------------------
    # Print progress every 1000 epochs
    # ----------------------------------------
    if (epoch + 1) % 1000 == 0:

        print(
            f"Epoch {(epoch + 1):5d} | "
            f"Data loss = {loss_d.item():.3e} | "
            f"Physics loss = {loss_p.item():.3e} | "
            f"Total training loss = {total_loss.item():.3e}"
        )

        print(
            f"        Coupling parameter K = k/m = {K_now.item():.5f}"
        )

**Check the Stage 2 fit**

After Stage 2 finishes, the following cell calculates the final training data loss, physics loss, total loss, and **held-out validation loss once**. The total loss is the training data loss plus `lambda_p` times the unweighted physics loss. The physics loss is estimated using a fresh sample of 512 collocation points, so it can differ from the last training-epoch value. Compare these values with the Stage 1 results. Does including the undamped ODE improve prediction on the unseen final 20%?

These evaluation values do not enter the training objective or change the trained parameters.


In [ ]:
# ============================================================
# STAGE 2: HELD-OUT VALIDATION (AFTER TRAINING ONLY)
# ============================================================
# One evaluation after all training epochs; no validation in optimizer.
model.eval()
with torch.no_grad():
    stage2_train_loss = data_loss().item()
    stage2_val_loss = validation_loss().item()
# Physics residuals require derivatives, even during evaluation.
with torch.enable_grad():
    stage2_physics_loss = physics_loss(n_physics=512).item()
stage2_total_loss = stage2_train_loss + lambda_p * stage2_physics_loss
model.train()
print("========================================")
print("STAGE 2: FINAL TRAINING AND VALIDATION LOSS")
print("========================================")
print(f"Data loss    = {stage2_train_loss:.6e}")
print(f"Physics loss = {stage2_physics_loss:.6e}")
print(f"Total loss   = {stage2_total_loss:.6e}")
print(f"Validation data loss = {stage2_val_loss:.6e}")


**Read the training-loss plots**

**Plot 1: Stage 1 — Training Data Loss**

In Stage 1, the network learns only from measured angles in the first 80% of the experiment. The plotted **training data loss** tells us how closely its predictions match those points. Lower values mean a closer fit to the training measurements.

**Plot 2: Stage 2 — Training Losses**

In Stage 2, we combine agreement with the data and agreement with the differential equations:

- **Data loss** $L_{\text{data}}$ measures the fit to training measurements.
- **Physics loss** $L_{\text{physics}}$ measures the squared undamped ODE residuals at collocation times.
- **Total PINN loss** $L_{\text{total}}=L_{\text{data}}+\lambda_pL_{\text{physics}}$ is what the optimizer minimizes. The weight $\lambda_p$ controls the influence of the physics term.

It is normal for the data loss and physics loss to change differently, because the network is balancing two goals.

**What about validation?** No validation measurements or validation-loss values influence the optimizer. After **each** stage finishes, the lesson computes one validation loss, combining left- and right-pendulum errors on the held-out 20%. Compare the two values to see whether adding physics helped.

An epoch is one optimization update. Compare the histories with the validation results reported after each completed stage.


In [ ]:
# ============================================================
# TRAINING LOSS PLOTS (NO VALIDATION DURING TRAINING)
# ============================================================

# Stage 1: fit to measured training points
plt.figure(figsize=(11, 5))
stage1_epochs = np.arange(1, len(warmup_history) + 1)
plt.semilogy(stage1_epochs, warmup_history,
             linewidth=1.8, label="Data loss")
plt.xlabel("Epoch")
plt.ylabel("Normalized data loss")
plt.title("Stage 1: Training Data Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Stage 2: data, physics and total losses
plt.figure(figsize=(11, 5))
stage2_epochs = np.arange(1, len(data_history) + 1)
plt.semilogy(stage2_epochs, total_history,
             linewidth=1.6, label="Total PINN loss")
plt.semilogy(stage2_epochs, physics_history,
             linewidth=1.6, label="Physics loss")
plt.semilogy(stage2_epochs, data_history,
             linewidth=1.6, label="Data loss")
plt.xlabel("Epoch")
plt.ylabel("Normalized loss")
plt.title("Stage 2: Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Why can the loss graphs look spiky?**

Do not expect every loss curve to decrease smoothly at every epoch. The Adam optimizer can move slightly up and down while searching for better parameters. In **Stage 2**, the physics loss is especially likely to fluctuate because the code draws a new random set of collocation points at every epoch. The logarithmic vertical scale also makes small fluctuations look more dramatic.

Focus on the **overall trend**, not on whether every individual step decreases. A decreasing training loss means the model is fitting the known data better; a decreasing physics loss means the ODE residuals are becoming smaller; and the final validation loss, computed after training, tells us how well it predicts the held-out measurements.

A low training loss does not guarantee a low held-out validation loss. We compare the validation values **after Stage 1 and Stage 2** to see how the completed models perform.

**Why can the total loss look close to the physics loss?**

In Stage 2, the total loss is

$$
L_{\text{total}}=L_{\text{data}}+\lambda_p L_{\text{physics}}.
$$

If the data loss is small compared with the **weighted physics loss**, then

$$
L_{\text{total}}\approx\lambda_p L_{\text{physics}}.
$$

However, the plot shows the **unweighted** physics loss $L_{\text{physics}}$. When $\lambda_p=1$, the total and physics curves can nearly overlap. When $\lambda_p=0.5$, the total loss is instead approximately $0.5L_{\text{physics}}$. These curves can still look close on a logarithmic plot, even though they are not equal.

> **❓ Question 4: How Much Should We Trust the Physics?**
>
> Compare $\lambda_p=0.0,\ 0.5,\ 2.0$ in $L_{\mathrm{total}}=L_{\mathrm{data}}+\lambda_pL_{\mathrm{physics}}$, keeping your chosen `N_FREQ` from Question 3.
>
> For each weight:
>
> 1. Rebuild the network and reset the physical parameter to $K=1.0$, then re-run Stage 1 and its validation check.
> 2. Set `lambda_p`, run Stage 2, and run its post-stage validation check.
> 3. Record the summary below and inspect the final trajectory plots for both pendulums.
>
> | $\lambda_p$ | Data loss | Physics loss (unweighted) | Total loss | Validation loss | Final $K$ |
> |---:|---:|---:|---:|---:|---:|
> | 0.0 |  |  |  |  |  |
> | 0.5 |  |  |  |  |  |
> | 2.0 |  |  |  |  |  |
>
> **Discuss your observations**
>
> 1. *How does increasing $\lambda_p$ change the balance between fitting data and satisfying the ODEs? Compare data loss with the weighted physics contribution, $\lambda_pL_{\mathrm{physics}}$.*
> 2. *Which weight would you choose based on validation loss and both trajectory plots? Does a smaller physics loss always give better predictions?*
> 3. *Why can comparing total losses across different weights be misleading?*
>
> **Before continuing:** Rebuild the network, reset $K$, and re-run both stages and their validation checks using your chosen weight. Keep this setting for Question 5.

## 6. Results and sensitivity to initial K

**Parameter history.** Inspect $K=k/m$ during Stage 2. A curve that levels off suggests stabilization; continued large changes suggest incomplete convergence. Estimating an ODE coefficient from observed motion is an **inverse problem**.


In [ ]:
# ============================================================
# PHYSICAL-PARAMETER HISTORY
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    K_history,
    label=r"$K=k/m$"
)

plt.xlabel("Epoch")
plt.ylabel(r"$k/m$ [s$^{-2}$]")
plt.title("Estimated Coupling Parameter")
plt.grid(True)
plt.legend()

plt.show()


**Final predictions and metrics**

After training, we evaluate the model at 3,000 closely spaced time values. This gives smooth curves for

$$
\widehat{\theta}_1(t)
\qquad\text{and}\qquad
\widehat{\theta}_2(t).
$$

We also compare the predictions with the training and validation measurements.

We have already computed a held-out validation loss **after Stage 1** and **after Stage 2**. Here we evaluate the final model again and report more detailed metrics. No validation loss was calculated during training or used to update the network.

Several numerical errors are reported. Two useful ones are:

**Root Mean Square Error (RMSE)**

RMSE measures a typical size of the prediction error. Smaller is better.

**Relative $L^2$ error**

This compares the overall size of the prediction error with the overall size of the measured signal.

The most important idea is not the name of each metric. Ask:

> How close is the predicted motion to the measurements that were not used directly for fitting?

Remember that the final 20% of measured angles were held out from the data-fitting loss.

The following left- and right-pendulum plots compare the smooth predictions with training and held-out measurements. Inspect both because their motion is coupled; pay particular attention to whether oscillations remain close to the measurements in the shaded validation region.


In [ ]:
# ============================================================
# FINAL PINN PREDICTIONS + VALIDATION METRICS
# ============================================================

t_test = np.linspace(
    t_min,
    t_max,
    3000,
    dtype=np.float32
).reshape(-1, 1)


t_test_torch = torch.tensor(
    t_test,
    dtype=torch.float32,
    device=device
)

model.eval()

with torch.no_grad():

    prediction = (
        model(t_test_torch)
        .cpu()
        .numpy()
    )

    pred_train = (
        model(t_train_torch)
        .cpu()
        .numpy()
    )

    pred_val = (
        model(t_val_torch)
        .cpu()
        .numpy()
    )


theta1_pinn = prediction[:, 0]
theta2_pinn = prediction[:, 1]

# ------------------------------------------------------------
# Final training and validation metrics
# ------------------------------------------------------------

train_loss_final = np.mean(
    ((pred_train[:, 0] - theta1_train[:, 0]) / theta_scale)**2
) + np.mean(
    ((pred_train[:, 1] - theta2_train[:, 0]) / theta_scale)**2
)

val_loss_final = np.mean(
    ((pred_val[:, 0] - theta1_val[:, 0]) / theta_scale)**2
) + np.mean(
    ((pred_val[:, 1] - theta2_val[:, 0]) / theta_scale)**2
)

rmse_train_left = np.sqrt(
    np.mean((pred_train[:, 0] - theta1_train[:, 0])**2)
)
rmse_train_right = np.sqrt(
    np.mean((pred_train[:, 1] - theta2_train[:, 0])**2)
)

rmse_val_left = np.sqrt(
    np.mean((pred_val[:, 0] - theta1_val[:, 0])**2)
)
rmse_val_right = np.sqrt(
    np.mean((pred_val[:, 1] - theta2_val[:, 0])**2)
)

relative_l2_train_left = (
    np.linalg.norm(pred_train[:, 0] - theta1_train[:, 0])
    /
    max(np.linalg.norm(theta1_train[:, 0]), 1e-12)
)

relative_l2_train_right = (
    np.linalg.norm(pred_train[:, 1] - theta2_train[:, 0])
    /
    max(np.linalg.norm(theta2_train[:, 0]), 1e-12)
)

relative_l2_val_left = (
    np.linalg.norm(pred_val[:, 0] - theta1_val[:, 0])
    /
    max(np.linalg.norm(theta1_val[:, 0]), 1e-12)
)

relative_l2_val_right = (
    np.linalg.norm(pred_val[:, 1] - theta2_val[:, 0])
    /
    max(np.linalg.norm(theta2_val[:, 0]), 1e-12)
)

print()
print("========================================")
print("FINAL TRAINING / VALIDATION PERFORMANCE")
print("========================================")
print(f"Normalized training loss   = {train_loss_final:.6e}")
print(f"Normalized validation loss = {val_loss_final:.6e}")
print()
print(f"Training RMSE (left pendulum)     = {rmse_train_left:.6e} rad")
print(f"Validation RMSE (left pendulum)   = {rmse_val_left:.6e} rad")
print(f"Training RMSE (right pendulum)    = {rmse_train_right:.6e} rad")
print(f"Validation RMSE (right pendulum)  = {rmse_val_right:.6e} rad")
print()
print(f"Training relative L2 error (left)    = {relative_l2_train_left:.6e}")
print(f"Validation relative L2 error (left)  = {relative_l2_val_left:.6e}")
print(f"Training relative L2 error (right)   = {relative_l2_train_right:.6e}")
print(f"Validation relative L2 error (right) = {relative_l2_val_right:.6e}")


In [ ]:
# ============================================================
# LEFT PENDULUM: TRAINING + VALIDATION + RESIDUALS
# ============================================================

# Main trajectory comparison
plt.figure(figsize=(12, 5))

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=12,
    alpha=0.25,
    label="Training measurements"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label="Validation measurements"
)

plt.plot(
    t_test[:, 0],
    theta1_pinn,
    color = "red",
    linewidth=2,
    label="PINN trajectory"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Left Pendulum: Held-Out Validation Points vs PINN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# RIGHT PENDULUM: TRAINING + VALIDATION + RESIDUALS
# ============================================================

# Main trajectory comparison
plt.figure(figsize=(12, 5))

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=12,
    alpha=0.25,
    label="Training measurements"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label="Validation measurements"
)


plt.plot(
    t_test[:, 0],
    theta2_pinn,
    color = "red",
    linewidth=2,
    label="PINN trajectory"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Right Pendulum: Held-Out Validation Points vs PINN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Stage comparison.** Compare the training data-loss histories and Stage 2 data, unweighted physics, and total losses with the separately reported post-stage validation values.


In [ ]:
# ============================================================
# END-OF-LESSON LOSS AND VALIDATION COMPARISONS
# Validation computed ONCE per stage, not at each epoch
# ============================================================

# Data loss over both stages
plt.figure(figsize=(11, 5))
plt.semilogy(np.arange(1, len(warmup_history) + 1),
             warmup_history, linewidth=1.6, label="Stage 1 training data loss")
plt.semilogy(np.arange(1, len(data_history) + 1),
             data_history, linewidth=1.6, label="Stage 2 training data loss")
plt.xlabel("Epoch within stage")
plt.ylabel("Normalized data loss")
plt.title("Training Data Loss: Stage 1 vs Stage 2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Physics-informed training loss components
stage2_epochs = np.arange(1, len(total_history) + 1)
plt.figure(figsize=(11, 5))
plt.semilogy(stage2_epochs, data_history, linewidth=1.6, label="Data loss")
plt.semilogy(stage2_epochs, physics_history, linewidth=1.6, label="Physics loss")
plt.semilogy(stage2_epochs, total_history, linewidth=1.6, label="Total PINN loss")
plt.xlabel("Epoch")
plt.ylabel("Normalized loss")
plt.title("Stage 2: Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Estimated coupling and takeaway.** Print the final $K=k/m$ in $\mathrm{s}^{-2}$. The PINN combines experimental data and differential equations to learn trajectories and an unknown coefficient. A close fit alone does not establish that the coefficient is physically correct.


In [ ]:
# ============================================================
# FINAL PARAMETER ESTIMATE
# ============================================================

K_final = physical_parameters()


print()
print("========================================")
print("FINAL ESTIMATED PARAMETER")
print("========================================")

print(
    "Coupling parameter K = k/m =",
    K_final.item(),
    "s^-2"
)


> **❓ Question 5: Do the Initial Parameter Guesses Matter?**
>
> Keep your chosen `N_FREQ` and $\lambda_p$ from Questions 3 and 4. Compare starting values $K_0=0.2,\ 1.0,\ 2.0$.
>
> For each starting value:
>
> 1. Rebuild the network and re-run Stage 1 and its validation check.
> 2. Change the value in the parameter-definition cell, for example:
>    ```python
>    raw_K = nn.Parameter(inverse_softplus(0.2))
>    ```
>    Re-run that cell **before creating the Stage 2 optimizer**.
> 3. Run Stage 2, its validation check, and the final prediction plots.
>
> | Initial $K_0$ | Final $K$ | Validation loss |
> |---:|---:|---:|
> | 0.2 |  |  |
> | 1.0 |  |  |
> | 2.0 |  |  |
>
> **Discuss your observations**
>
> *How sensitive are the final $K$ and validation loss to the starting guess? Use your results to assess consistency, and discuss whether similar estimates alone establish the true physical value.*

